# Computational Complexity of Vocoders

We report the number of multiplications required to generate 1 second long 22050 Hz audio with different neural vocoders. We do not report real time factors as they are implementation dependent. In the calculation, We do not consider the complexity of mathematical functions, computation happening in feature upsampling and source signal generation. A N point FFT is assumed to cost $2N\log_2 N$ real multiplications. 

In [1]:
from math import ceil, log2
fft = lambda x: (2 * x * log2(x))

## b-NSF

b-NSF is basically 5 10-layer WaveNets stacked together. The WaveNets have 64 residual channels and 128 skip channels. 

In [2]:
b_nsf_total = (1 * 64 + (64 * 128 * 3 + 64 + 64 * 64 + 64 * 128) * 10 + 128 * 2) * 5 * 22050
print(f"b-NSF costs {b_nsf_total/10**10:0.0f} * 10^10 multiplications per second.")

b-NSF costs 4 * 10^10 multiplications per second.


## Gaussian WaveNet

Parameters for Gaussian WaveNet is 128 skip channels, 64 residual channels, 24 layers convolutions with kernel size of 3.

In [3]:
wavenet_total = (1 * 64 + 24 * (64 * 128 * 3 + 64 * 64 + 64 + 64 * 128) + 128 * 128 + 128 * 2) * 22050
print(f"Gaussian WaveNet costs {wavenet_total/10**10:0.0f} * 10^10 multiplications per second.")

Gaussian WaveNet costs 2 * 10^10 multiplications per second.


## Parallel WaveGAN

Parallel WaveGAN is basically a 30 layer wavenet with 64 skip channels, and 64 residual channels.

In [4]:
pwg_total = (1 * 64 + 30 * (64 * 128 * 3 + 64 * 64 + 64 + 64 * 64) + 64 * 64 + 64 * 1) * 22050
print(f"Parallel WaveGAN costs {pwg_total/10**10:0.0f} * 10^10 multiplications per second.")

Parallel WaveGAN costs 2 * 10^10 multiplications per second.


## LPCNet

We use the formula given in the paper (originally counting FLOPs). The first GRU is made sparse in LPCNet. 
$C=\left(3 d N_{A}^{2}+3 N_{B}\left(N_{A}+N_{B}\right)+2 N_{B} Q\right) \cdot f_{s}$, where $d=0.1$, $N_{A}=384$, $N_{B}=16$, and $Q=256$, $f_s = 22050$. 

In [5]:
lpc_total = (3 * 0.1 * 384 ** 2 + 3 * 16 * (384 + 16) + 2 * 16 * 256) * 22050
print(f"LPCNet costs {lpc_total/10**9:0.1f} * 10^9 multiplications per second.")

LPCNet costs 1.6 * 10^9 multiplications per second.


## NHV

The total multiplications in two convolutional neural networks:

In [6]:
nhv_convs = (80 * 3 * 256 + (256 / 8) * 3 * (256 / 8) * 8 * 2 + 256 * 3 * 222) * 2 * 22050 / 128
print(f"CNNs in NHV costs {nhv_convs/10**8:0.1f} * 10^8 multiplications per second.")

CNNs in NHV costs 1.0 * 10^8 multiplications per second.


Following steps are necessary for LTV filtering in the NHV model in frame $m$, $0\le m < M - 1$. We carry out the convolutions in frequency domain with FFT.
  1. Convert $\hat h_n$ and $\hat h_h$ to DFT of impulse responses
  2. Calculate DFT of $x[n] w_L[n-mL]$ in each frame
  3. Convolution in frequency domain by multiplication of FFT coefficients. Then use IDFT and OLA to obtain $s[n]$
  4. The last 50ms FIR can be implemented with blocked FFT and OLA.  
  
Total multiplications in LTV filtering:

In [7]:
nhv_ffts = (fft(1024 + 128) * 2 + (1024 + 128) * 4 + fft(1024 + 128)) * 2 * 22050 / 128 + 22050 / 1024 * fft(2048)
print(f"LTV filtering in NHV costs {nhv_ffts/10**8:0.1f} * 10^8 multiplications per second.")

LTV filtering in NHV costs 0.3 * 10^8 multiplications per second.


In [8]:
nhv_total = nhv_convs + nhv_ffts
print(f"NHV costs {nhv_total/10**8:0.1f} * 10^8 multiplications per second.")

NHV costs 1.2 * 10^8 multiplications per second.


## Griffin-Lim Iteration

Assume the FFT size is 1024, frame shift is 256 sampling points, and iteration number is 50.

In [9]:
gl_total = fft(1024) * 22050 / 256 * 2 * 50
print(f"GL costs {gl_total/10**8:0.1f} * 10^8 multiplications per second.")

GL costs 1.8 * 10^8 multiplications per second.
